In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, when, lower, regexp_replace, concat_ws, explode, count, size
from pyspark.ml.feature import Tokenizer, StopWordsRemover

spark = SparkSession.builder.appName("AmazonReviews").getOrCreate()

def process_data(file_path):
    df = spark.read.text(file_path)

    split_col = split(col('value'), ' ', 2)

    df = df.withColumn('split_col', split_col)
    df = df.filter(size(col('split_col')) == 2)

    df = df.withColumn('label', col('split_col').getItem(0)) \
           .withColumn('reviewText', col('split_col').getItem(1)) \
           .drop('split_col', 'value')

    df = df.withColumn('sentiment', when(col('label') == '__label__1', 'Negative').otherwise('Positive'))

    df = df.withColumn('clean_text', lower(col('reviewText'))) \
           .withColumn('clean_text', regexp_replace(col('clean_text'), r'[^a-z\s]', ''))

    tokenizer = Tokenizer(inputCol="clean_text", outputCol="words")
    df = tokenizer.transform(df)

    remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
    df = remover.transform(df)

    df = df.withColumn('cleaned_review', concat_ws(' ', col('filtered_words')))

    return df.select('sentiment', 'cleaned_review')

train_path = '/content/train.ft.txt'
test_path = '/content/test.ft.txt'

train_df = process_data(train_path)
test_df = process_data(test_path)

combined_df = train_df.union(test_df)

print("--- Sample of Cleaned Data ---")
combined_df.show(5, truncate=False)

words_df = combined_df.select('sentiment', explode(split(col('cleaned_review'), ' ')).alias('word'))
words_df = words_df.filter(col('word') != '')

positive_words = words_df.filter(col('sentiment') == 'Positive') \
                         .groupBy('word') \
                         .agg(count('word').alias('frequency')) \
                         .orderBy(col('frequency').desc())

print("--- Top 10 words in POSITIVE reviews ---")
positive_words.show(10)

negative_words = words_df.filter(col('sentiment') == 'Negative') \
                         .groupBy('word') \
                         .agg(count('word').alias('frequency')) \
                         .orderBy(col('frequency').desc())

print("--- Top 10 words in NEGATIVE reviews ---")
negative_words.show(10)

--- Sample of Cleaned Data ---
+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|sentiment|cleaned_review                                                                                                                                                                                                                                                                                                                                                                                                     

In [ ]:
from pyspark.ml.feature import NGram
from pyspark.sql.functions import split, explode, count, col, trim, regexp_replace

temp_df = combined_df.withColumn("cleaned_review", trim(regexp_replace(col("cleaned_review"), r"\s+", " ")))

temp_df = temp_df.withColumn("words_array", split(col("cleaned_review"), " "))

ngram = NGram(n=2, inputCol="words_array", outputCol="bigrams")
bigrams_df = ngram.transform(temp_df)

exploded_bigrams = bigrams_df.select('sentiment', explode(col('bigrams')).alias('bigram'))
exploded_bigrams = exploded_bigrams.withColumn("bigram", trim(col("bigram")))
exploded_bigrams = exploded_bigrams.filter((col('bigram') != '') & (col('bigram') != ' '))

positive_bigrams = exploded_bigrams.filter(col('sentiment') == 'Positive') \
                         .groupBy('bigram') \
                         .agg(count('bigram').alias('frequency')) \
                         .orderBy(col('frequency').desc())

print("--- Top 10 Bigrams in POSITIVE reviews ---")
positive_bigrams.show(10, truncate=False)

negative_bigrams = exploded_bigrams.filter(col('sentiment') == 'Negative') \
                         .groupBy('bigram') \
                         .agg(count('bigram').alias('frequency')) \
                         .orderBy(col('frequency').desc())

print("--- Top 10 Bigrams in NEGATIVE reviews ---")
negative_bigrams.show(10, truncate=False)

--- Top 10 Bigrams in POSITIVE reviews ---
+----------------+---------+
|bigram          |frequency|
+----------------+---------+
|read book       |12044    |
|one best        |11531    |
|highly recommend|10060    |
|great book      |9068     |
|year old        |6636     |
|years ago       |5955     |
|good book       |5659     |
|recommend book  |5291     |
|first time      |4730     |
|works great     |4710     |
+----------------+---------+
only showing top 10 rows
--- Top 10 Bigrams in NEGATIVE reviews ---
+----------------+---------+
|bigram          |frequency|
+----------------+---------+
|waste money     |16280    |
|waste time      |10944    |
|dont waste      |10147    |
|dont buy        |9161     |
|read book       |8147     |
|much better     |7686     |
|dont know       |7259     |
|save money      |6326     |
|year old        |5290     |
|customer service|4612     |
+----------------+---------+
only showing top 10 rows


In [ ]:
import pandas as pd
import plotly.express as px

sentiment_counts = combined_df.groupBy('sentiment').count().toPandas()

pos_pd = positive_bigrams.limit(10).toPandas()
neg_pd = negative_bigrams.limit(10).toPandas()

fig_pie = px.pie(sentiment_counts,
                 values='count',
                 names='sentiment',
                 title='Overall Customer Satisfaction',
                 color='sentiment',
                 color_discrete_map={'Positive':'#00CC96', 'Negative':'#EF553B'})
fig_pie.show()

fig_pos = px.bar(pos_pd,
                 x='frequency',
                 y='bigram',
                 orientation='h',
                 title='Top 10 Patterns in Positive Reviews',
                 color_discrete_sequence=['#00CC96'])
fig_pos.update_layout(yaxis={'categoryorder':'total ascending'})
fig_pos.show()

fig_neg = px.bar(neg_pd,
                 x='frequency',
                 y='bigram',
                 orientation='h',
                 title='Top 10 Patterns in Negative Reviews',
                 color_discrete_sequence=['#EF553B'])
fig_neg.update_layout(yaxis={'categoryorder':'total ascending'})
fig_neg.show()

pos_pd.to_csv('positive_patterns.csv', index=False)
neg_pd.to_csv('negative_patterns.csv', index=False)
sentiment_counts.to_csv('sentiment_counts.csv', index=False)
